# Présidentielles françaises 2002-2022 : analyse et tendance 2027

Notebook principal du projet. Il suit le processus KDD :
1. Sélection des données
2. Prétraitement
3. Transformation
4. Fouille (analyse, tendance 2027)
5. Interprétation

Le code est organisé en fonctions, appelées depuis `main()` à la fin du notebook.
Les données brutes doivent être placées dans `data/raw/` (voir le README).

In [ ]:
from pathlib import Path
import json
import re
import unicodedata
import zipfile

import numpy as np

import pandas as pd
import plotly.express as px
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
import pyarrow.parquet as pq

DOSSIER_PROJET = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DOSSIER_RAW = DOSSIER_PROJET / "data" / "raw"
DOSSIER_REFERENCE = DOSSIER_PROJET / "data" / "reference"
DOSSIER_PROCESSED = DOSSIER_PROJET / "data" / "processed"

ANNEES = [2002, 2007, 2012, 2017, 2022]
ELECTIONS = [f"{annee}_pres_t{tour}" for annee in ANNEES for tour in (1, 2)]

## 1. Sélection des données

In [ ]:
def charger_resultats_generaux(chemin=DOSSIER_RAW / "general_results.parquet", elections=ELECTIONS):
    """Charge les résultats généraux (inscrits, votants, abstention...) des présidentielles, par bureau de vote."""
    table = pq.read_table(chemin, filters=[("id_election", "in", elections)])
    return table.to_pandas()


def charger_resultats_candidats(chemin=DOSSIER_RAW / "candidats_results.parquet", elections=ELECTIONS):
    """Charge les voix par candidat et par bureau de vote pour les présidentielles."""
    colonnes = ["id_election", "code_departement", "code_commune", "code_bv",
                "nom", "prenom", "voix", "ratio_voix_inscrits", "ratio_voix_exprimes"]
    table = pq.read_table(chemin, columns=colonnes, filters=[("id_election", "in", elections)])
    return table.to_pandas()


def charger_familles(chemin=DOSSIER_REFERENCE / "candidats_familles.csv"):
    """Charge la table qui associe chaque candidat à une famille politique."""
    return pd.read_csv(chemin)

## 2. Prétraitement

In [ ]:
def normaliser_nom(nom):
    """Met un nom en majuscules sans accents, pour comparer les années entre elles."""
    sans_accents = unicodedata.normalize("NFKD", str(nom)).encode("ascii", "ignore").decode()
    return sans_accents.upper().strip()


def ajouter_annee_tour(df):
    """Ajoute les colonnes annee et tour à partir de id_election (ex. 2022_pres_t1)."""
    df = df.copy()
    df["annee"] = df["id_election"].str[:4].astype(int)
    df["tour"] = df["id_election"].str[-1].astype(int)
    return df


def ajouter_familles(candidats, familles):
    """Associe chaque ligne de candidat à sa famille politique (jointure sur l'année et le nom)."""
    familles = familles.assign(cle_nom=familles["nom"].map(normaliser_nom))[["annee", "cle_nom", "famille"]]
    candidats = candidats.assign(cle_nom=candidats["nom"].map(normaliser_nom))
    resultat = candidats.merge(familles, on=["annee", "cle_nom"], how="left").drop(columns="cle_nom")
    manquants = resultat.loc[resultat["famille"].isna(), "nom"].unique()
    if len(manquants) > 0:
        print("Candidats sans famille politique :", list(manquants))
    return resultat

## 3. Transformation

In [ ]:
def agreger_par_departement(generaux):
    """Somme les résultats généraux par élection et par département, puis recalcule les taux."""
    colonnes = ["inscrits", "abstentions", "votants", "blancs", "nuls", "exprimes"]
    dep = (generaux.groupby(["id_election", "annee", "tour", "code_departement", "libelle_departement"])[colonnes]
           .sum().reset_index())
    dep["taux_abstention"] = dep["abstentions"] / dep["inscrits"]
    dep["taux_blancs_nuls"] = (dep["blancs"] + dep["nuls"]) / dep["votants"]
    return dep


def voix_par_famille_departement(candidats, generaux_dep):
    """Calcule la part des exprimés obtenue par chaque famille politique, par élection et par département."""
    voix = (candidats.groupby(["id_election", "annee", "tour", "code_departement", "famille"])["voix"]
            .sum().reset_index())
    voix = voix.merge(generaux_dep[["id_election", "code_departement", "exprimes"]],
                      on=["id_election", "code_departement"], how="left")
    voix["part_exprimes"] = voix["voix"] / voix["exprimes"]
    return voix

## Cartes

Les contours des départements de métropole viennent de [france-geojson](https://github.com/gregoiredavid/france-geojson). L'outre-mer et les Français de l'étranger ne sont pas sur la carte.

In [ ]:
def charger_contours(chemin=DOSSIER_REFERENCE / "departements.geojson"):
    """Charge les contours des départements de métropole (GeoJSON, propriété code = code du département)."""
    with open(chemin, encoding="utf-8") as f:
        return json.load(f)


def carte_departements(df, colonne, titre, contours=None):
    """Dessine une carte choroplèthe des départements pour une colonne de df (indexée par code_departement)."""
    contours = contours or charger_contours()
    codes_metropole = {f["properties"]["code"] for f in contours["features"]}
    df = df[df["code_departement"].isin(codes_metropole)]
    fig = px.choropleth(df, geojson=contours, locations="code_departement",
                        featureidkey="properties.code", color=colonne,
                        hover_name="code_departement", color_continuous_scale="Viridis",
                        title=titre)
    fig.update_geos(fitbounds="locations", visible=False)
    fig.update_layout(margin=dict(l=0, r=0, t=40, b=0))
    return fig

## 5. Tendance 2027

Méthode :
1. Pour chaque département et chaque famille politique, on calcule l'**écart** entre le score du département et le score national, au 1er tour de chaque présidentielle (2002 à 2022).
2. On prolonge cet écart jusqu'en 2027 avec une régression linéaire (pente calculée sur les cinq élections).
3. On ajoute cet écart au score national de la famille dans la **moyenne des sondages 2027** (`sondages.json` : chaque hypothèse compte pareil dans son sondage, chaque sondage compte pareil).
4. Les scores négatifs sont ramenés à 0, puis chaque département est remis à 100 %.

C'est une **tendance**, pas une prévision : elle suppose que la géographie du vote continue d'évoluer comme entre 2002 et 2022.
On la compare ensuite à la projection publiée avec les sondages (site Repère 2027).

In [ ]:
def lire_csv_sondage(nom_fichier):
    """Lit un CSV des fichiers sondages 2027 (séparateur ; et virgule décimale)."""
    return pd.read_csv(DOSSIER_RAW / nom_fichier, sep=";", decimal=",", encoding="utf-8-sig",
                       dtype={"code_departement": str})


def charger_sondages(chemin=DOSSIER_RAW / "sondages.json"):
    """Charge les sondages 2027 (une ligne par sondage, hypothèse et candidat)."""
    with open(chemin, encoding="utf-8") as f:
        brut = json.load(f)
    lignes = []
    for i, sondage in enumerate(brut["sondages"]):
        for j, hypothese in enumerate(sondage["hypotheses"]):
            for slug, score in hypothese["scores"].items():
                lignes.append({"id_sondage": i, "institut": sondage["institut"],
                               "terrain_fin": sondage["terrain_fin"], "id_hypothese": j,
                               "slug": slug, "intentions_pct": score})
    return pd.DataFrame(lignes)


def moyenne_sondages(sondages, chemin=DOSSIER_REFERENCE / "candidats_familles_2027.csv"):
    """Moyenne des sondages : chaque hypothèse compte pareil dans son sondage, chaque sondage compte pareil.

    Renvoie les intentions moyennes par candidat (avec sa famille) et par famille (en proportion, total = 1)."""
    table = pd.read_csv(chemin)
    sondages = sondages.merge(table[["slug", "nom", "famille"]], on="slug", how="left")
    manquants = sondages.loc[sondages["famille"].isna(), "slug"].unique()
    if len(manquants) > 0:
        print("Candidats 2027 sans famille :", list(manquants))
    # Grille complète : un candidat absent d'une hypothèse y compte pour 0.
    grille = sondages.pivot_table(index=["id_sondage", "id_hypothese"], columns="slug",
                                  values="intentions_pct", fill_value=0)
    par_sondage = grille.groupby(level="id_sondage").mean()
    candidats = par_sondage.mean().rename("intentions_pct").reset_index()
    candidats = candidats.merge(table[["slug", "nom", "famille"]], on="slug", how="left")
    candidats["cle_nom"] = candidats["nom"].map(normaliser_nom)
    familles = candidats.groupby("famille")["intentions_pct"].sum()
    print(f"{sondages['id_sondage'].nunique()} sondages, {len(grille)} hypothèses, "
          f"du {sondages['terrain_fin'].min()} au {sondages['terrain_fin'].max()}")
    return candidats, familles / familles.sum()


def parts_premier_tour(familles_dep, generaux_dep):
    """Table complète (département x famille x année) des parts au 1er tour, avec 0 quand une famille n'a pas de candidat."""
    t1 = familles_dep[familles_dep["tour"] == 1]
    grille = pd.MultiIndex.from_product([t1["code_departement"].unique(), t1["famille"].unique(), ANNEES],
                                        names=["code_departement", "famille", "annee"])
    parts = t1.set_index(["code_departement", "famille", "annee"])["part_exprimes"].reindex(grille, fill_value=0)
    return parts.reset_index()


def parts_nationales(candidats, generaux):
    """Part nationale de chaque famille au 1er tour de chaque présidentielle."""
    t1 = candidats[candidats["tour"] == 1]
    voix = t1.groupby(["annee", "famille"])["voix"].sum()
    exprimes = generaux[generaux["tour"] == 1].groupby("annee")["exprimes"].sum()
    return (voix / exprimes.reindex(voix.index.get_level_values("annee")).values).rename("part_nationale").reset_index()


def tendance_2027(parts_dep, parts_nat, intentions_famille, annee_cible=2027):
    """Prolonge l'écart département/national de chaque famille et l'applique au sondage 2027."""
    df = parts_dep.merge(parts_nat, on=["annee", "famille"], how="left").fillna({"part_nationale": 0})
    df["ecart"] = df["part_exprimes"] - df["part_nationale"]

    def prolonger(groupe):
        pente, origine = np.polyfit(groupe["annee"], groupe["ecart"], 1)
        dernier = groupe.loc[groupe["annee"].idxmax(), "ecart"]
        return pd.Series({"ecart_2022": dernier, "pente_par_an": pente,
                          "ecart_2027": dernier + pente * (annee_cible - groupe["annee"].max())})

    ecarts = df.groupby(["code_departement", "famille"]).apply(prolonger, include_groups=False).reset_index()
    ecarts["intention_nationale"] = ecarts["famille"].map(intentions_famille).fillna(0)
    ecarts["part_brute"] = (ecarts["intention_nationale"] + ecarts["ecart_2027"]).clip(lower=0)
    ecarts.loc[ecarts["intention_nationale"] == 0, "part_brute"] = 0
    total = ecarts.groupby("code_departement")["part_brute"].transform("sum")
    ecarts["part_2027"] = ecarts["part_brute"] / total
    return ecarts.drop(columns="part_brute")


CANDIDATS_PROJETES = {"MARINE LE PEN": ("LE PEN", "extreme_droite"),
                      "EDOUARD PHILIPPE": ("PHILIPPE", "centre"),
                      "JEAN-LUC MELENCHON": ("MELENCHON", "gauche_radicale")}


def comparer_projection_site(tendance, sondage):
    """Compare notre tendance à la projection départementale du site pour les 3 candidats qu'il projette."""
    site = lire_csv_sondage("projection-departementale-1er-tour.csv")
    infos = site["candidat"].map(normaliser_nom).map(CANDIDATS_PROJETES)
    site["cle_nom"], site["famille"] = infos.str[0], infos.str[1]
    # Poids du candidat dans sa famille (sondage national), pour passer de la famille au candidat.
    sondage = sondage.assign(poids=sondage["intentions_pct"] / sondage.groupby("famille")["intentions_pct"].transform("sum"))
    site = site.merge(sondage[["cle_nom", "poids"]], on="cle_nom", how="left")
    site = site.merge(tendance[["code_departement", "famille", "part_2027"]], on=["code_departement", "famille"], how="left")
    site["notre_estimation_pct"] = 100 * site["part_2027"] * site["poids"]
    resume = site.groupby("candidat").apply(lambda g: pd.Series({
        "correlation": g["score_estime_pct"].corr(g["notre_estimation_pct"]),
        "ecart_moyen_points": (g["notre_estimation_pct"] - g["score_estime_pct"]).abs().mean()}),
        include_groups=False)
    return site, resume

## 6. Contexte socio-économique

Sources reliées aux résultats par département :
- **Communes 2026** (data.gouv.fr) : population, superficie et type d'espace (urbain ou rural) de chaque commune.
- **Filosofi 2023** (INSEE) : niveau de vie, pauvreté, inégalités, par département. Une seule année : c'est une photo, pas une évolution.
- **ACLED** (version agrégée) : manifestations et émeutes par région et par semaine, depuis 2020. Le niveau est la région, donc chaque département reçoit la valeur de sa région.
- **Indice des prix** (INSEE, série 001759970) : contexte national, inflation sur un an avant chaque élection.

In [ ]:
INDICATEURS_FILOSOFI = {"MED_SL": "niveau_vie_median", "PR_MD60": "taux_pauvrete",
                        "GI_SL": "indice_gini", "S_EI_DI_UNE": "part_revenu_chomage",
                        "S_SOC_BEN_DI_MIN_SOC": "part_revenu_minima_sociaux",
                        "S_RET_PEN_DI": "part_revenu_retraites"}


def charger_communes(chemin=DOSSIER_RAW / "communes-france-2026.parquet"):
    """Charge la liste des communes 2026 (code INSEE, département, région, population, superficie, type d'espace)."""
    colonnes = ["code_insee", "dep_code", "reg_nom", "population", "superficie_km2",
                "grille_densite_texte", "latitude_centre", "longitude_centre"]
    return pd.read_parquet(chemin, columns=colonnes)


def profil_departements(communes):
    """Résume les communes par département : population, densité, part de la population en commune rurale."""
    communes = communes.assign(rural=communes["grille_densite_texte"].str.startswith(("Rural", "Bourgs")))
    profil = communes.groupby("dep_code").agg(
        reg_nom=("reg_nom", "first"), population=("population", "sum"),
        superficie_km2=("superficie_km2", "sum"),
        population_rurale=("population", lambda p: p[communes.loc[p.index, "rural"]].sum()))
    profil["densite"] = profil["population"] / profil["superficie_km2"]
    profil["part_population_rurale"] = profil["population_rurale"] / profil["population"]
    return profil.drop(columns="population_rurale").reset_index().rename(columns={"dep_code": "code_departement"})


def charger_filosofi(chemin=DOSSIER_RAW / "FILOSOFI_CC_csv.zip"):
    """Charge les indicateurs Filosofi 2023 au niveau département (une colonne par indicateur)."""
    with zipfile.ZipFile(chemin) as z:
        nom = next(n for n in z.namelist() if n.endswith("_data.csv"))
        with z.open(nom) as f:
            brut = pd.read_csv(f, sep=";", dtype={"GEO": str})
    dep = brut[(brut["GEO_OBJECT"] == "DEP") & brut["FILOSOFI_MEASURE"].isin(INDICATEURS_FILOSOFI)]
    table = dep.pivot_table(index="GEO", columns="FILOSOFI_MEASURE", values="OBS_VALUE")
    return table.rename(columns=INDICATEURS_FILOSOFI).reset_index().rename(columns={"GEO": "code_departement"})


def charger_acled(chemin=DOSSIER_RAW / "acled_france_2020_2026.csv"):
    """Charge l'export ACLED agrégé (événements par région, semaine et type), France seulement."""
    acled = pd.read_csv(chemin, parse_dates=["week"])
    acled = acled[acled["country"] == "France"].copy()
    acled["cle_region"] = acled["admin1"].map(normaliser_nom)
    return acled


def acled_par_region(acled, debut, fin, types=("Protests", "Riots")):
    """Compte les événements (manifestations, émeutes) par région entre deux dates."""
    periode = acled[(acled["week"] >= debut) & (acled["week"] < fin) & acled["event_type"].isin(types)]
    table = periode.pivot_table(index="cle_region", columns="event_type", values="events",
                                aggfunc="sum", fill_value=0)
    return table.rename(columns=lambda t: "evenements_" + t.lower()).reset_index()


def charger_ipc(chemin=DOSSIER_RAW / "serie_001759970_23112026.zip"):
    """Charge l'indice des prix à la consommation mensuel (France) et calcule l'inflation sur 12 mois."""
    with zipfile.ZipFile(chemin) as z:
        with z.open("valeurs_mensuelles.csv") as f:
            ipc = pd.read_csv(f, sep=";", skiprows=4, header=None, usecols=[0, 1], names=["mois", "ipc"])
    ipc["mois"] = pd.to_datetime(ipc["mois"])
    ipc = ipc.sort_values("mois").set_index("mois")
    ipc["inflation_12_mois"] = ipc["ipc"].pct_change(12)
    return ipc


def table_departements(familles_dep, generaux_dep, profil, filosofi, acled_region, election="2022_pres_t1"):
    """Assemble une ligne par département : vote par famille, abstention, profil, Filosofi et ACLED."""
    vote = (familles_dep[familles_dep["id_election"] == election]
            .pivot_table(index="code_departement", columns="famille", values="part_exprimes", fill_value=0)
            .add_prefix("vote_"))
    abstention = generaux_dep[generaux_dep["id_election"] == election].set_index("code_departement")["taux_abstention"]
    table = vote.join(abstention).reset_index()
    table = table.merge(profil, on="code_departement", how="inner").merge(filosofi, on="code_departement", how="left")
    table["cle_region"] = table["reg_nom"].map(normaliser_nom)
    table = table.merge(acled_region, on="cle_region", how="left").drop(columns="cle_region")
    population_region = table.groupby("reg_nom")["population"].transform("sum")
    for col in [c for c in table.columns if c.startswith("evenements_")]:
        table[col + "_pour_100k"] = table[col] / population_region * 1e5
    return table

## 7. Analyse

- Évolution nationale de chaque famille politique et de l'abstention, de 2002 à 2022, prolongée par la moyenne des sondages 2027.
- Départements où une famille a le plus progressé entre deux élections.
- Modèle explicatif : régression linéaire du vote par département sur les indicateurs socio-économiques (variables standardisées, pour comparer les coefficients).

Attention : les indicateurs sont corrélés entre eux (niveau de vie, pauvreté, inégalités), donc le signe d'un coefficient isolé est à lire avec prudence. Le R² indique la part des différences entre départements que le modèle explique.

In [ ]:
def evolution_nationale(parts_nat, intentions_famille):
    """Table année x famille des parts nationales au 1er tour, avec 2027 (moyenne des sondages)."""
    table = parts_nat.pivot_table(index="annee", columns="famille", values="part_nationale", fill_value=0)
    table.loc[2027] = intentions_famille.reindex(table.columns).fillna(0)
    return table


def graphique_evolution(evolution, familles=None):
    """Courbes de l'évolution nationale des familles politiques (2027 = sondages)."""
    familles = familles or list(evolution.columns)
    df = evolution[familles].reset_index().melt(id_vars="annee", var_name="famille", value_name="part")
    fig = px.line(df, x="annee", y="part", color="famille", markers=True,
                  title="Part des exprimés au 1er tour (2027 : moyenne des sondages)")
    fig.add_vline(x=2024.5, line_dash="dot")
    fig.update_yaxes(tickformat=".0%")
    return fig


def evolution_abstention(generaux_dep):
    """Taux d'abstention national par élection et par tour."""
    nat = generaux_dep.groupby(["annee", "tour"])[["abstentions", "inscrits"]].sum()
    return (nat["abstentions"] / nat["inscrits"]).unstack("tour").rename(columns=lambda t: f"tour_{t}")


def departements_qui_bougent(parts_dep, famille, annee_debut=2017, annee_fin=2022, n=5):
    """Départements de métropole où une famille a le plus gagné et le moins gagné entre deux élections (en points)."""
    metropole = ~parts_dep["code_departement"].str.startswith("Z")
    df = parts_dep[metropole & (parts_dep["famille"] == famille) & parts_dep["annee"].isin([annee_debut, annee_fin])]
    table = df.pivot_table(index="code_departement", columns="annee", values="part_exprimes")
    table["evolution_points"] = 100 * (table[annee_fin] - table[annee_debut])
    table = table.sort_values("evolution_points")
    return pd.concat([table.tail(n)[::-1], table.head(n)])


def modele_explicatif(departements, cible, indicateurs):
    """Régression linéaire standardisée : quels indicateurs expliquent le vote d'une famille par département ?"""
    donnees = departements[[cible] + indicateurs].dropna()
    X = StandardScaler().fit_transform(donnees[indicateurs])
    modele = LinearRegression().fit(X, donnees[cible])
    coefficients = pd.Series(modele.coef_ * 100, index=indicateurs, name="effet_en_points").sort_values()
    return modele.score(X, donnees[cible]), coefficients

## 4. Contrôles

In [ ]:
def resume_national(candidats):
    """Affiche les quatre premiers candidats au niveau national pour chaque élection (contrôle de cohérence)."""
    totaux = candidats.groupby(["id_election", "nom"])["voix"].sum().reset_index()
    return (totaux.sort_values(["id_election", "voix"], ascending=[True, False])
            .groupby("id_election").head(4))

## Main

In [ ]:
def main():
    generaux = ajouter_annee_tour(charger_resultats_generaux())
    candidats = ajouter_annee_tour(charger_resultats_candidats())
    candidats = ajouter_familles(candidats, charger_familles())

    print(f"Résultats généraux : {len(generaux):,} lignes")
    print(f"Résultats candidats : {len(candidats):,} lignes")

    generaux_dep = agreger_par_departement(generaux)
    familles_dep = voix_par_famille_departement(candidats, generaux_dep)

    DOSSIER_PROCESSED.mkdir(parents=True, exist_ok=True)
    generaux_dep.to_parquet(DOSSIER_PROCESSED / "participation_departements.parquet", index=False)
    familles_dep.to_parquet(DOSSIER_PROCESSED / "familles_departements.parquet", index=False)

    display(resume_national(candidats))

    exemple = familles_dep[(familles_dep["id_election"] == "2022_pres_t1")
                           & (familles_dep["famille"] == "extreme_droite")]
    carte_departements(exemple, "part_exprimes",
                       "Extrême droite au 1er tour 2022 (part des exprimés)").show()
    sondage, intentions_famille = moyenne_sondages(charger_sondages())
    print("Intentions 2027 par famille (moyenne des sondages) :")
    display(intentions_famille.sort_values(ascending=False))

    parts_dep = parts_premier_tour(familles_dep, generaux_dep)
    parts_nat = parts_nationales(candidats, generaux)
    tendance = tendance_2027(parts_dep, parts_nat, intentions_famille)
    tendance.to_parquet(DOSSIER_PROCESSED / "tendance_2027_departements.parquet", index=False)

    site, comparaison = comparer_projection_site(tendance, sondage)
    print("Comparaison avec la projection du site (par département) :")
    display(comparaison)

    ed = tendance[tendance["famille"] == "extreme_droite"]
    carte_departements(ed, "part_2027", "Tendance 2027 : extrême droite au 1er tour (part des exprimés)").show()
    # Contexte socio-économique (élection 2022, ACLED sur l'année avant le 1er tour du 10 avril 2022)
    acled_region = acled_par_region(charger_acled(), "2021-04-10", "2022-04-10")
    departements = table_departements(familles_dep, generaux_dep, profil_departements(charger_communes()),
                                      charger_filosofi(), acled_region)
    departements.to_parquet(DOSSIER_PROCESSED / "departements_2022.parquet", index=False)
    print(f"Table départements 2022 : {departements.shape[0]} départements, {departements.shape[1]} colonnes")
    indicateurs = ["niveau_vie_median", "taux_pauvrete", "indice_gini", "part_revenu_chomage",
                   "densite", "part_population_rurale", "evenements_protests_pour_100k"]
    print("Corrélations (départements, 1er tour 2022) :")
    display(departements[["vote_extreme_droite", "vote_gauche_radicale", "vote_centre", "taux_abstention"] + indicateurs]
            .corr().loc[indicateurs, ["vote_extreme_droite", "vote_gauche_radicale", "vote_centre", "taux_abstention"]]
            .round(2))

    ipc = charger_ipc()
    print("Inflation sur 12 mois en mars de chaque année d'élection :")
    display(ipc.loc[[f"{a}-03-01" for a in ANNEES], "inflation_12_mois"].round(3))
    # Analyse
    evolution = evolution_nationale(parts_nat, intentions_famille)
    evolution.to_parquet(DOSSIER_PROCESSED / "evolution_nationale.parquet")
    graphique_evolution(evolution).show()
    print("Abstention nationale :")
    display(evolution_abstention(generaux_dep).round(3))
    print("Extrême droite, 2017 → 2022 : plus fortes hausses et plus faibles (points) :")
    display(departements_qui_bougent(parts_dep, "extreme_droite")["evolution_points"].round(1))
    for cible in ["vote_extreme_droite", "vote_gauche_radicale", "vote_centre", "taux_abstention"]:
        r2, coefficients = modele_explicatif(departements, cible, indicateurs)
        print(f"{cible} : R² = {r2:.2f} ; effet d'un écart-type de chaque indicateur (points) :")
        display(coefficients.round(2))
    return generaux_dep, familles_dep, tendance, departements


if __name__ == "__main__":
    generaux_dep, familles_dep, tendance, departements = main()